# Flan-T5 Instruction Fine-Tuning (Colab + T4)

Run this on a **T4 GPU runtime** in Google Colab.
The repo is cloned automatically so all source files are available.

In [ ]:
# --- CLONE REPO ---
import sys
from pathlib import Path

REPO_URL = "https://github.com/vietnq/flan_t5"  # <-- change to your repo URL
PROJECT_DIR = Path("/content/flan_t5")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already cloned. Pulling latest...")
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Project root: {PROJECT_DIR}")

In [ ]:
# --- INSTALL DEPENDENCIES ---
!pip install -q torch transformers datasets accelerate sentencepiece tensorboard rouge-score nltk sacrebleu tqdm rich pyyaml

In [ ]:
# --- CONFIG ---
# Change this to point to any config in configs/
CONFIG_PATH = "configs/exp33_cot/flan_t5_small_with_cot.yaml"
BASE_CONFIG_PATH = None  # auto-resolves to configs/base.yaml

# --- MEMORY SETTINGS ---
# Dataset now uses HuggingFace streaming — only the requested samples
# are downloaded. MAX_SAMPLES controls how many records to fetch.
# 1000 = ~6MB RAM. None = full dataset (~20K samples, ~120MB RAM).
MAX_SAMPLES = 1000  # total samples before train/eval split
MAX_TRAIN_SAMPLES = None  # further limit after split
MAX_EVAL_SAMPLES = None

OUTPUT_DIR = None  # set to override auto-generated path

print(f"Config: {CONFIG_PATH}")
print(f"Max samples: {MAX_SAMPLES}")

In [ ]:
import gc
import torch

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

free_memory()

In [ ]:
from src.utils.config import load_config
from src.utils.helpers import set_seed
from src.utils.logging import (
    get_output_dir,
    log_config_summary,
    save_config,
    save_metrics,
    print_results_table,
    setup_logging,
)

config = load_config(CONFIG_PATH, BASE_CONFIG_PATH)
set_seed(config.get("advanced", {}).get("seed", 42))

output_dir = OUTPUT_DIR or get_output_dir(config)
logger = setup_logging(output_dir=output_dir)

log_config_summary(config, logger)
save_config(config, output_dir)

print(f"Output dir: {output_dir}")

In [ ]:
from src.utils.helpers import get_device_info

device_info = get_device_info()
print(f"Device: {device_info['device']}")
if "gpu_name" in device_info:
    print(f"GPU: {device_info['gpu_name']} ({device_info['gpu_memory_gb']:.1f} GB)")

In [ ]:
from src.models.model import load_model_and_tokenizer
from src.utils.helpers import count_parameters

print("Loading model and tokenizer...")
model, tokenizer = load_model_and_tokenizer(config)

param_counts = count_parameters(model)
total = param_counts["total"]
trainable = param_counts["trainable"]
print(f"Model parameters: {total:,} total, {trainable:,} trainable")

free_memory()

In [ ]:
from src.data.dataset import load_datasets

print("Loading datasets...")
# max_samples filters early, before full JSON is processed
datasets = load_datasets(config, max_samples=MAX_SAMPLES)
train_dataset = datasets["train"]
eval_dataset = datasets["eval"]

print(f"Train samples: {len(train_dataset):,}")
print(f"Eval samples: {len(eval_dataset):,}")

if MAX_TRAIN_SAMPLES:
    train_dataset = train_dataset.select(range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))
if MAX_EVAL_SAMPLES:
    eval_dataset = eval_dataset.select(range(min(MAX_EVAL_SAMPLES, len(eval_dataset))))

free_memory()

In [ ]:
from src.data.preprocessing import tokenize_dataset
from src.utils.helpers import get_device

print("Tokenizing datasets...")
data_config = config["data"]
device = get_device()
num_proc = data_config.get("preprocessing_num_workers", 4)

train_dataset = tokenize_dataset(
    train_dataset,
    tokenizer,
    max_source_length=data_config.get("max_source_length", 512),
    max_target_length=data_config.get("max_target_length", 256),
    use_cot=data_config.get("use_cot", True),
    num_proc=num_proc,
)
eval_dataset = tokenize_dataset(
    eval_dataset,
    tokenizer,
    max_source_length=data_config.get("max_source_length", 512),
    max_target_length=data_config.get("max_target_length", 256),
    use_cot=data_config.get("use_cot", True),
    num_proc=num_proc,
)

print(f"Train tokenized: {len(train_dataset)} samples")
print(f"Eval tokenized: {len(eval_dataset)} samples")

free_memory()

In [ ]:
from src.evaluation.metrics import compute_metrics_with_tokenizer
from src.training.trainer import create_trainer

print("Creating trainer...")
trainer = create_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    config=config,
    output_dir=output_dir,
    compute_metrics_fn=compute_metrics_with_tokenizer(tokenizer),
)

print("Trainer ready.")

In [ ]:
# --- TRAIN ---
# On a T4 with flan-t5-small, 1000 samples takes ~10-15 min.
# Full dataset (~20K samples) would take ~3-5h.

print("Starting training...")
resume = config.get("advanced", {}).get("resume_from_checkpoint")
train_result = trainer.train(resume_from_checkpoint=resume)
print("Training complete.")

free_memory()

In [ ]:
# --- SAVE FINAL MODEL ---
print("Saving final model and tokenizer...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
save_metrics(metrics, output_dir, split="train")
print(f"Model saved to {output_dir}")

In [ ]:
# --- FINAL EVALUATION ---
print("Running final evaluation...")
eval_result = trainer.evaluate()
trainer.log_metrics("eval", eval_result)
trainer.save_metrics("eval", eval_result)
save_metrics(eval_result, output_dir, split="eval")

print_results_table(eval_result, title="Final Evaluation Results")
print(f"Outputs saved to: {output_dir}")